# jaraco.context Package Testing

This notebook tests the jaraco.context package, which provides context management utilities including directory management and exception suppression.

## Example 1: Basic Context Manager Usage

In [ ]:
import jaraco.context
import tempfile
import os

# Test suppress context manager 
print("Testing suppress context manager:")
with jaraco.context.suppress(ValueError):
    print("This will execute")
    
with jaraco.context.suppress(ValueError):
    print("About to raise ValueError...")
    raise ValueError("This error will be suppressed")
    print("This won't execute")

print("After suppressed exception")

# Test temporary directory context
print("\nTesting temporary directory:")
with jaraco.context.temp_dir() as tmp:
    print(f"Temporary directory: {tmp}")
    
    # Create a test file
    test_file = os.path.join(tmp, "test.txt")
    with open(test_file, 'w') as f:
        f.write("Hello from temp dir")
    
    print(f"Created file: {test_file}")
    print(f"File exists: {os.path.exists(test_file)}")

print("Temporary directory cleaned up automatically")

## Example 2: Directory Context Operations

In [ ]:
import jaraco.context
import os
import tempfile

# Test pushd (push directory) context manager
current_dir = os.getcwd()
print(f"Current directory: {current_dir}")

# Create a temporary directory for testing
with tempfile.TemporaryDirectory() as temp_dir:
    print(f"\nUsing temporary directory: {temp_dir}")
    
    # Change to temp directory using pushd
    with jaraco.context.pushd(temp_dir):
        print(f"Inside pushd context: {os.getcwd()}")
        
        # Create a subdirectory
        subdir = "subdir"
        os.makedirs(subdir)
        
        # Change to subdirectory
        with jaraco.context.pushd(subdir):
            print(f"Inside nested pushd: {os.getcwd()}")
            
            # Create a file
            with open("nested_file.txt", 'w') as f:
                f.write("In nested directory")
        
        print(f"Back in temp dir: {os.getcwd()}")

print(f"Back in original directory: {os.getcwd()}")
print(f"Directories match: {os.getcwd() == current_dir}")

## Example 3: Exception Context Handling

In [ ]:
import jaraco.context

# Test exception handling contexts
def risky_operation(should_fail=False):
    if should_fail:
        raise RuntimeError("Operation failed!")
    return "Success!"

print("Testing exception contexts:")

# Test with successful operation: suppress() leaves a body that does not raise alone
with jaraco.context.suppress(RuntimeError):
    result = risky_operation(False)
    print(f"Result: {result}")
assert result == "Success!", result

# Test with failing operation: suppress() swallows the RuntimeError and skips the rest of its body (measured)
reached = []
with jaraco.context.suppress(RuntimeError):
    result = risky_operation(True)
    reached.append(result)
    print(f"This won't print: {result}")
assert reached == [], reached
print("Exception was suppressed successfully")

# Test without suppression: the same call raises
try:
    result = risky_operation(True)
    print(f"Won't reach here: {result}")
except RuntimeError as e:
    assert str(e) == "Operation failed!", e
    print(f"Without suppress(), the RuntimeError propagates as expected ({e})")
else:
    raise AssertionError(f"risky_operation(True) returned {result!r} instead of raising")


## Example 4: Working Directory Management

In [ ]:
import jaraco.context
import os
import tempfile
from pathlib import Path

# Test working directory context management
print("Testing working directory management:")

original_cwd = Path.cwd()
print(f"Original working directory: {original_cwd}")

# Create a test directory structure
with tempfile.TemporaryDirectory() as base_temp:
    base_path = Path(base_temp)
    
    # Create subdirectories
    dir1 = base_path / "dir1"
    dir2 = base_path / "dir2" / "subdir"
    dir1.mkdir()
    dir2.mkdir(parents=True)
    
    print(f"\nCreated test directories:")
    print(f"  {dir1}")
    print(f"  {dir2}")
    
    # Test multiple directory changes
    with jaraco.context.pushd(dir1):
        print(f"\nIn dir1: {Path.cwd()}")
        
        # Create a file
        Path("file1.txt").write_text("Content 1")
        files = list(Path.cwd().glob("*"))
        print(f"Files in dir1: {files}")
        
        with jaraco.context.pushd(dir2):
            print(f"\nIn dir2: {Path.cwd()}")
            Path("file2.txt").write_text("Content 2")
            files = list(Path.cwd().glob("*"))
            print(f"Files in dir2: {files}")
        
        print(f"\nBack in dir1: {Path.cwd()}")

print(f"\nBack in original directory: {Path.cwd()}")
print(f"Directory restored correctly: {Path.cwd() == original_cwd}")

## Example 5: Context Utility Combinations

In [ ]:
import jaraco.context
import tempfile
import os

# Test combining multiple context utilities
def complex_operation():
    print("Starting complex operation...")
    
    # Use suppress and temp_dir together
    with jaraco.context.suppress(FileNotFoundError), \
         jaraco.context.temp_dir() as tmp_dir:
        
        print(f"Working in temporary directory: {tmp_dir}")
        
        # Use pushd within the temp directory
        with jaraco.context.pushd(tmp_dir):
            print(f"Current directory: {os.getcwd()}")
            
            # Create some test files
            for i in range(3):
                filename = f"test_{i}.txt"
                with open(filename, 'w') as f:
                    f.write(f"Content for file {i}")
                print(f"Created {filename}")
            
            # Try to read a file that doesn't exist (will be suppressed)
            try:
                with open("nonexistent.txt", 'r') as f:
                    content = f.read()
            except FileNotFoundError:
                print("FileNotFoundError was handled by suppress")
            
            # List all files created
            files = [f for f in os.listdir('.') if f.endswith('.txt')]
            print(f"Files created: {files}")
            
            return files

result = complex_operation()
print(f"\nOperation completed. Files were: {result}")
print("All contexts properly cleaned up")